In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Instalação da Biblioteca

In [ ]:
%pip install pyarrow

Convertendo arquivos .txt em .parquet

In [ ]:
from pathlib import Path
import json
import pandas as pd

input_dir = Path(
    "/content/drive/MyDrive/TCC_Chatbot/uploads/"
)

output_dir = input_dir / "parquet"

output_dir.mkdir(exist_ok=True)

for file_path in input_dir.glob("*.txt"):

    print(f"Convertendo {file_path.name}...")

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)

        df = pd.DataFrame(data)

        output_path = output_dir / f"{file_path.stem}.parquet"

        df.to_parquet(
            output_path,
            index=False,
            engine="pyarrow",
            compression="snappy"
        )

        print(
            f"{output_path.name} criado "
            f"({len(df)} registros)"
        )

    except Exception as error:
        print(f"Erro em {file_path.name}: {error}")

print("\nTodos os arquivos foram processados!")

Concatenação dos parquets em só arquivo

In [ ]:
import pandas as pd
import os
import glob

input_dir = (
    "/content/drive/MyDrive/TCC_Chatbot/uploads/parquet"
)

parquet_files = glob.glob(
    os.path.join(input_dir, "*.parquet")
)

print("Arquivos encontrados:", len(parquet_files))

dataframes = []

for file_path in parquet_files:

    print("Carregando:", file_path)

    df = pd.read_parquet(file_path)

    dataframes.append(df)

final_dataset = pd.concat(
    dataframes,
    ignore_index=True
)

print("\nAntes da remoção de duplicatas:")
print(final_dataset.shape)

final_dataset = (
    final_dataset
    .drop_duplicates(
        subset=["question", "answer"]
    )
    .reset_index(drop=True)
)

print("\nDepois da remoção de duplicatas:")
print(final_dataset.shape)

output_path = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/python_docs.parquet"
)

final_dataset.to_parquet(
    output_path,
    index=False
)

print("\nSalvo em:", output_path)

Verificação da estrutura do dataset criado

In [ ]:
import pandas as pd

file_path = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/python_docs.parquet"
)

df = pd.read_parquet(file_path)

print(df.columns)
df.info()
print(df.shape)
print(df.head())

Adicionando a coluna language - pt

In [ ]:
import pandas as pd

file_path = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/python_docs.parquet"
)

df = pd.read_parquet(file_path)

df["language"] = "pt"

df.to_parquet(
    file_path,
    index=False
)

print("Coluna 'language' adicionada com sucesso!")
print(df.head())
print("\nIdiomas:")
print(df["language"].value_counts())